# Stap 4 - hoekmeting en PID
Gebruik `voer_pid_stap4_uit` met `Stap_4_PID.ino`. Hieronder stel je de PID af.

Verderop staat **stap 5: PWM-veranderingsbegrenzing**. Daar gebruik je `voer_pid_stap5_uit` met `Stap_5_PWM_begrenzing.ino`.


In [1]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit, voer_pid_stap5_uit
from closed_loop.figuren import plot_pid_meting
import matplotlib.pyplot as plt
com_poort = "COM4"  # Vul de poort van jouw Arduino in.


### Ruwe hoek meten

Laat de motorvoeding uit. Geef meetduur en delta_t in seconden op. De uitvoer toont Arduino-tijd vanaf het eerste meetpunt en de ruwe sensorhoek. PWM is nul. De meting wordt als CSV opgeslagen in data/. Noteer de hoek in de nulstand; de correctie voegen we in de volgende stap toe.


In [2]:
hoekmetingen = test_hoek(com_poort=com_poort, meetduur=1.0, delta_t=0.1)


PWM: 0; stabiliseren: 0 s
Meting gestart. Onderbreek de notebookcel om te stoppen.
   0.000 s |  115.576 graden
   0.100 s |  115.488 graden
   0.200 s |  115.488 graden
   0.300 s |  115.488 graden
   0.400 s |  115.488 graden
   0.500 s |  115.488 graden
   0.600 s |  115.576 graden
   0.700 s |  115.576 graden
   0.800 s |  115.576 graden
   0.900 s |  115.488 graden
STOP verzonden naar de Arduino.
CSV opgeslagen: C:\Users\jbusink1\OneDrive - HvA\Documenten\GitHub\Practica HvA Github\practicum2_2\Closed_loop\data\aeropendulum_20261102_175529_708535.csv
Aantal metingen: 10


### 2. PID instellen en meten

Begin met ki = 0 en kd = 0. Vul de gemeten beginhoek in. De werkhoek is sensorhoek - beginhoek (omgerekend naar -180..180). Controleer dat de werkhoek toeneemt als de motor de arm omhoog beweegt. De PID berekent de PWM; max_pwm begrenst deze van 0 tot en met 255 (0 houdt de motor uit). Doelhoek mag van -180 tot 180 graden zijn, gemeten vanaf stilstand. Bij beginhoek 89 en sensorhoek 89 is de werkhoek 0; bij sensorhoek 0 is de werkhoek 89 graden, ongeveer 90. Een doelhoek van 90 betekent een verplaatsing van 90 graden vanaf stilstand, niet een ruwe sensorwaarde van 90. Met toon_metingen kies je of de meetpunten live worden geprint. Fixed stopt na meetduur; continuous loopt tot je de cel onderbreekt. Delta T bepaalt het meet- en regelinterval.


In [ ]:
# Stap 4: upload Stap_4_PID.ino.
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 3
ki = 3
kd = 3
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True  # False: geen meetpunten printen

metingen, csv_pad = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode
)

### 3. Resultaat bekijken en CSV inlezen

De PID-meting wordt opgeslagen in een CSV met alle instellingen bovenaan (regels die met `#` beginnen), gevolgd door `tijd_s,hoek_deg,pwm`. Er wordt geen apart JSON-bestand meer aangemaakt. Tijden in de instellingen met `_ms` zijn in milliseconden; `tijd_s` in de meetdata is in seconden.

De cel hieronder leest de laatst opgeslagen PID-meting in. Vervang `csv_pad` door een bestandspad om een andere meting te openen. Voor een veiligheidsproef gebruik je `csv_pad_veiligheid`. Deze uitleescel is bedoeld voor het nieuwe CSV-formaat.


In [4]:
import csv
import pandas as pd

bestand = csv_pad  # Of bijvoorbeeld "data/pid_20261027_....csv"

# Lees de instellingen uit de commentaarheader.
with open(bestand, encoding="utf-8", newline="") as f:
    instellingen = {}
    for rij in csv.reader(f):
        if not rij or not rij[0].startswith("# "):
            break
        instellingen[rij[0][2:]] = rij[1]

# Pandas slaat de commentaarheader over en leest de meetkolommen.
df = pd.read_csv(bestand, comment="#")
display(pd.Series(instellingen, name="Instelling").to_frame())
display(df.head())


,Instelling
com_poort,COM4
doelhoek,90.0
kp,3
ki,3
kd,3
max_pwm,255
meetduur_ms,10000
sensorcontrole,False
regelinterval_ms,20
mode,fixed


,tijd_s,hoek_deg,pwm
0,0.02,0.088,255.0
1,0.04,0.088,255.0
2,0.06,0.000,255.0
3,0.08,0.088,255.0
4,0.10,0.440,242.0


### 4. De maximale hoek aanpassen

Gebruik dezelfde functie en dezelfde geuploade sketch. Tot nu toe hoefde je geen `max_hoek` mee te geven: de functie gebruikt standaard 180 graden. Geef nu `max_hoek=120` mee. Bij overschrijding van de ingestelde werkhoek stopt de Arduino de motor en de meting. De doelhoek moet binnen de ingestelde grens liggen.


In [ ]:
# Vervolg: dezelfde sketch en functie, nu met een lagere stopgrens.
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 4
ki = 3
kd = 0
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True 
max_hoek = 100.0  # Stopgrens voor de werkhoek; kies passend bij de opstelling.

metingen_veiligheid, csv_pad_veiligheid = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek, max_hoek=max_hoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode,
)


## Stap 5 - maximale PWM-sprong per regelstap

Gebruik `voer_pid_stap5_uit`. Zonder `max_pwm_stap` wordt geen begrenzer ingesteld. Met `max_pwm_stap=20` mag de werkelijk aangeboden PWM per regelstap maximaal 20 telwaarden stijgen; dalen gaat direct. Kleine veranderingen worden direct overgenomen. Een vraag van 0 naar 255 wordt bijvoorbeeld 0, 20, 40, 60, enzovoort. STOP en andere stops zetten de uitgang direct op nul.

Bij een ander regelinterval verandert ook de maximale verandering per seconde. `max_pwm` blijft de maximale uitgang; bij 0 draait de motor niet.

De begrenzer beperkt uitsluitend positieve PWM-sprongen. Een verlaging, ook naar nul, wordt direct overgenomen.


In [1]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit, voer_pid_stap5_uit
import pandas as pd
import matplotlib.pyplot as plt

instellingen = dict(
    com_poort="COM4",
    beginhoek=115.4,  # Vul je gemeten ruwe sensorhoek in rust in.
    doelhoek=60.0,
    kp=4.5, ki=1.0, kd=3.0,  # Neem je eigen stap-4-instelling over.
    max_pwm=255,  # Eerst communicatie testen; daarna je afgesproken PWM-grens.
    meetduur=10.0,
    delta_t=0.02,
    mode="fixed",
    toon_metingen=True,
)

metingen_zonder, csv_zonder = voer_pid_stap5_uit(
    **instellingen
)


Stap 5: max_pwm=255, limiter=uit telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek    0.088 graden | doel   60.000 graden | PWM 255 (100.0%)
   0.040 s | hoek    0.000 graden | doel   60.000 graden | PWM 255 (100.0%)
   0.060 s | hoek    0.000 graden | doel   60.000 graden | PWM 255 (100.0%)
   0.080 s | hoek    0.176 graden | doel   60.000 graden | PWM 248 ( 97.3%)
   0.100 s | hoek    0.440 graden | doel   60.000 graden | PWM 234 ( 91.8%)
   0.120 s | hoek    0.791 graden | doel   60.000 graden | PWM 221 ( 86.7%)
   0.140 s | hoek    1.670 graden | doel   60.000 graden | PWM 139 ( 54.5%)
   0.160 s | hoek    2.901 graden | doel   60.000 graden | PWM  82 ( 32.2%)
   0.180 s | hoek    4.483 graden | doel   60.000 graden | PWM  23 (  9.0%)
   0.200 s | hoek    6.504 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.220 s | hoek    8.702 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.240 s | hoek   11.338 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.260 s | h

In [12]:
max_pwm_stap = 40
metingen_met, csv_met = voer_pid_stap5_uit(
    **instellingen, max_pwm_stap=max_pwm_stap,
)


Stap 5: max_pwm=255, limiter=40 telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek  -22.763 graden | doel   60.000 graden | PWM  40 ( 15.7%)
   0.040 s | hoek  -23.027 graden | doel   60.000 graden | PWM  80 ( 31.4%)
   0.060 s | hoek  -23.115 graden | doel   60.000 graden | PWM 120 ( 47.1%)
   0.080 s | hoek  -22.588 graden | doel   60.000 graden | PWM 160 ( 62.7%)
   0.100 s | hoek  -21.709 graden | doel   60.000 graden | PWM 200 ( 78.4%)
   0.120 s | hoek  -20.478 graden | doel   60.000 graden | PWM 187 ( 73.3%)
   0.140 s | hoek  -18.808 graden | doel   60.000 graden | PWM 116 ( 45.5%)
   0.160 s | hoek  -16.787 graden | doel   60.000 graden | PWM  55 ( 21.6%)
   0.180 s | hoek  -14.238 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.200 s | hoek  -11.162 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.220 s | hoek   -7.910 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.240 s | hoek   -4.394 graden | doel   60.000 graden | PWM   0 (  0.0%)
   0.260 s | ho